# Feature Engineering & Model Selection Techniques

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/akekapong78/ai-competition/blob/main/01-forecasting/feature_and_model_techniques.ipynb)

**เทคนิคที่จะเรียนใน notebook นี้:**

| # | เทคนิค | ทำไมนิยม |
|---|--------|----------|
| 1 | Fourier Features | จับ seasonality ซับซ้อนได้โดยไม่ต้อง guess period |
| 2 | SHAP Values | อธิบาย model ได้ว่า feature ไหนส่งผลยังไง |
| 3 | Permutation Importance | feature importance ที่ fair กว่า built-in |
| 4 | XGBoost vs LightGBM | เปรียบเทียบ 2 โมเดลยอดนิยม |
| 5 | Optuna | auto hyperparameter tuning ที่ดีที่สุดตอนนี้ |
| 6 | TimeSeriesSplit CV | cross-validation ถูกต้องสำหรับ time series |

In [ ]:
!pip install xgboost lightgbm shap optuna -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
import lightgbm as lgb
import shap
import optuna
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.inspection import permutation_importance

optuna.logging.set_verbosity(optuna.logging.WARNING)
plt.rcParams['figure.figsize'] = (13, 4)
print('Ready')

In [ ]:
# Dataset เดิม — synthetic solar
idx = pd.date_range('2023-01-01', '2024-12-31', freq='1h')
np.random.seed(42)
h = idx.hour
solar = np.maximum(0,
    10 * np.sin(np.pi*(h-6)/12)
    * (1 + 0.3*np.cos(2*np.pi*idx.month/12))
    + np.random.normal(0, 0.3, len(idx))
)
df = pd.DataFrame({'solar_mw': solar}, index=idx)
print(f'Dataset: {len(df):,} rows')

---
## Technique 1: Fourier Features

**ปัญหาของ `hour` feature ธรรมดา:**  
`hour=23` กับ `hour=0` ห่างกัน 23 ในสายตา model แต่จริงๆ ต่างกันแค่ 1 ชั่วโมง

**Fourier แก้ด้วย:** แปลง hour → คู่ sin/cos → บอก model ว่า pattern วนซ้ำ

```
hour=0  → sin=0.0,  cos=1.0
hour=6  → sin=1.0,  cos=0.0
hour=12 → sin=0.0,  cos=-1.0
hour=18 → sin=-1.0, cos=0.0
hour=23 → sin≈-0.26, cos≈0.97  ← ใกล้ hour=0 อีกครั้ง ✓
```

**เพิ่มความซับซ้อน:** ใส่ harmonics หลาย order จับ pattern ซับซ้อนขึ้น

In [ ]:
def fourier_features(index, period, n_harmonics):
    """
    สร้าง Fourier features สำหรับ seasonality
    
    period     : วงรอบ (24 = รายวัน, 168 = รายสัปดาห์, 8760 = รายปี)
    n_harmonics: ยิ่งมาก ยิ่งจับ pattern ซับซ้อน (แต่ระวัง overfit)
    """
    t = np.arange(len(index))
    features = {}
    for k in range(1, n_harmonics + 1):
        features[f'sin_{period}_{k}'] = np.sin(2 * np.pi * k * t / period)
        features[f'cos_{period}_{k}'] = np.cos(2 * np.pi * k * t / period)
    return pd.DataFrame(features, index=index)


def make_features_v2(df):
    """Feature engineering พร้อม Fourier"""
    d = df.copy()
    t = d['solar_mw']

    # Fourier: daily (24h), weekly (168h), yearly (8760h)
    d = pd.concat([
        d,
        fourier_features(d.index, period=24,   n_harmonics=3),  # รายวัน 3 harmonics
        fourier_features(d.index, period=168,  n_harmonics=2),  # รายสัปดาห์
        fourier_features(d.index, period=8760, n_harmonics=2),  # รายปี
    ], axis=1)

    # Basic time features
    d['is_day']     = ((d.index.hour >= 6) & (d.index.hour <= 18)).astype(int)
    d['solar_elev'] = np.maximum(0, np.sin(np.pi * (d.index.hour - 6) / 12))

    # Lag features
    for lag in [1, 2, 6, 12, 24, 48, 168]:
        d[f'lag_{lag}'] = t.shift(lag)

    # Rolling features
    d['roll_mean_24'] = t.shift(1).rolling(24).mean()
    d['roll_std_24']  = t.shift(1).rolling(24).std()
    d['roll_max_24']  = t.shift(1).rolling(24).max()

    return d.dropna()


data = make_features_v2(df)
FEATS = [c for c in data.columns if c != 'solar_mw']
print(f'Total features: {len(FEATS)}')
fourier_feats = [f for f in FEATS if 'sin_' in f or 'cos_' in f]
print(f'Fourier features: {fourier_feats}')

In [ ]:
# Visualize Fourier vs raw hour
sample = data.iloc[:48]  # 2 วัน
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(sample.index, sample['sin_24_1'], label='sin_24_1 (1st harmonic)')
axes[0].plot(sample.index, sample['sin_24_2'], label='sin_24_2 (2nd harmonic)', ls='--')
axes[0].plot(sample.index, sample['sin_24_3'], label='sin_24_3 (3rd harmonic)', ls=':')
axes[0].set_title('Fourier Features (Daily)')
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

axes[1].plot(sample.index, sample['solar_mw'], color='orange', label='Solar actual')
axes[1].set_title('Solar Power (2 days)')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()
print('สังเกต: sin_24_1 มี shape คล้าย solar curve มาก → feature ดี')

---
## Technique 2: TimeSeriesSplit Cross-Validation

**ห้ามใช้ random CV กับ time series** — data รั่วจากอนาคตไปอดีต

```
Random CV (ผิด):              TimeSeriesSplit (ถูก):
Fold 1: train=● test=○        Fold 1: [train===] [test=]
●○●●○●○○●●○●○●○              Fold 2: [train=====] [test=]
          ↑ leak!             Fold 3: [train=======] [test=]
```

In [ ]:
tscv = TimeSeriesSplit(n_splits=4, gap=24)  # gap=24h ป้องกัน data leak ข้ามวัน

X, y = data[FEATS], data['solar_mw']

# Visualize splits
fig, ax = plt.subplots(figsize=(13, 3))
for i, (train_idx, test_idx) in enumerate(tscv.split(X)):
    ax.barh(i, len(train_idx), left=0, color='steelblue', alpha=0.7, label='Train' if i==0 else '')
    ax.barh(i, len(test_idx), left=train_idx[-1], color='orange', alpha=0.7, label='Test' if i==0 else '')
ax.set_xlabel('Hours')
ax.set_title('TimeSeriesSplit — 4 Folds (train ขยายทีละ fold)')
ax.legend(); plt.tight_layout(); plt.show()

# CV score
cv_scores = []
for train_idx, test_idx in tscv.split(X):
    m = xgb.XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.05,
                          subsample=0.8, random_state=42, n_jobs=-1, verbosity=0)
    m.fit(X.iloc[train_idx], y.iloc[train_idx])
    pred = np.maximum(0, m.predict(X.iloc[test_idx]))
    cv_scores.append(mean_absolute_error(y.iloc[test_idx], pred))

print(f'CV MAE per fold: {[f"{s:.3f}" for s in cv_scores]}')
print(f'Mean CV MAE: {np.mean(cv_scores):.3f} ± {np.std(cv_scores):.3f} MW')

---
## Technique 3: SHAP Values

**Feature importance แบบ built-in** → บอกแค่ว่า feature ไหนใช้บ่อย  
**SHAP** → บอกว่า feature ส่งผลต่อ prediction **ทิศทางไหน** และ **มากแค่ไหน** ต่อแต่ละ sample

```
Built-in importance:   lag_168 = 0.15   ← แค่ตัวเลข
SHAP:                  lag_168 สูง → ดัน prediction ขึ้น +2.3 MW
                       lag_168 ต่ำ → ดัน prediction ลง -1.1 MW
```

ใช้ใน competition เพื่อ: debug model, เลือก features ที่ meaningful, presentable ใน pitching

In [ ]:
# Train final model
train = data[data.index < '2024-06-01']
test  = data[data.index >= '2024-06-01']

model = xgb.XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.05,
                          subsample=0.8, colsample_bytree=0.8, random_state=42,
                          n_jobs=-1, verbosity=0)
model.fit(train[FEATS], train['solar_mw'])

# SHAP — คำนวณ
explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(test[FEATS].iloc[:500])  # sample 500 rows เร็วขึ้น

print('SHAP values shape:', shap_values.shape)

In [ ]:
# SHAP Summary Plot — ดีที่สุดสำหรับเข้าใจ model
# แกน X = SHAP value (บวก = ดัน prediction ขึ้น)
# สี = ค่า feature (แดง=สูง, น้ำเงิน=ต่ำ)
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, test[FEATS].iloc[:500],
                  feature_names=FEATS, show=False)
plt.title('SHAP Summary — ผลกระทบของแต่ละ feature ต่อ prediction')
plt.tight_layout(); plt.show()

In [ ]:
# SHAP Waterfall — อธิบาย prediction 1 row (ทำให้ใจ pitching)
# "ทำไม model ถึง predict ว่า solar = X MW ณ เวลานี้"
sample_idx = 100
explanation = shap.Explanation(
    values    = shap_values[sample_idx],
    base_values = explainer.expected_value,
    data      = test[FEATS].iloc[sample_idx].values,
    feature_names = FEATS
)
plt.figure(figsize=(10, 6))
shap.waterfall_plot(explanation, show=False)
plt.title(f'SHAP Waterfall — prediction ณ {test.index[sample_idx]}')
plt.tight_layout(); plt.show()

---
## Technique 4: Permutation Importance

**ปัญหาของ built-in feature importance:**  
XGBoost นับว่า feature ถูกใช้บ่อยแค่ไหน → features ที่มี high cardinality (เช่น lag_1) มักได้ score สูงเสมอแม้ไม่ได้ดีจริง

**Permutation Importance:**  
สลับค่า feature นั้นแบบ random → ดูว่า model error เพิ่มแค่ไหน  
ถ้า error เพิ่มมาก = feature นั้นสำคัญจริง

In [ ]:
perm = permutation_importance(
    model, test[FEATS], test['solar_mw'],
    n_repeats=10, random_state=42, n_jobs=-1,
    scoring='neg_mean_absolute_error'
)

perm_df = pd.DataFrame({
    'feature':    FEATS,
    'importance': perm.importances_mean,
    'std':        perm.importances_std
}).sort_values('importance', ascending=False)

# เปรียบเทียบ built-in vs permutation
builtin_df = pd.DataFrame({
    'feature':    FEATS,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

builtin_df.head(12).plot(kind='barh', x='feature', y='importance',
                          ax=ax1, color='steelblue', legend=False)
ax1.set_title('Built-in Feature Importance')
ax1.invert_yaxis()

perm_df.head(12).plot(kind='barh', x='feature', y='importance',
                       ax=ax2, color='darkorange', legend=False,
                       xerr=perm_df.head(12)['std'].values)
ax2.set_title('Permutation Importance (more reliable)')
ax2.invert_yaxis()

plt.tight_layout(); plt.show()
print('สังเกตว่า ranking เปลี่ยนไหม — ถ้าเปลี่ยนมาก แสดงว่า built-in misleading')

---
## Technique 5: XGBoost vs LightGBM

| | XGBoost | LightGBM |
|-|---------|----------|
| **Speed** | ช้ากว่า | เร็วกว่า 10x บน data ใหญ่ |
| **Memory** | ใช้มาก | ประหยัดกว่า |
| **Small data** | ดีกว่าเล็กน้อย | ดี |
| **Category features** | ต้อง encode เอง | handle ได้ native |
| **API** | คล้ายกันมาก | คล้ายกันมาก |

**แนะนำ competition:** train ทั้งสอง → ensemble (average prediction) → มักดีกว่าตัวเดียว

In [ ]:
import time

# XGBoost
t0 = time.time()
xgb_model = xgb.XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.05,
                               subsample=0.8, colsample_bytree=0.8,
                               random_state=42, n_jobs=-1, verbosity=0)
xgb_model.fit(train[FEATS], train['solar_mw'])
xgb_time = time.time() - t0
xgb_pred = np.maximum(0, xgb_model.predict(test[FEATS]))

# LightGBM
t0 = time.time()
lgb_model = lgb.LGBMRegressor(n_estimators=300, max_depth=6, learning_rate=0.05,
                                subsample=0.8, colsample_bytree=0.8,
                                random_state=42, n_jobs=-1, verbose=-1)
lgb_model.fit(train[FEATS], train['solar_mw'])
lgb_time = time.time() - t0
lgb_pred = np.maximum(0, lgb_model.predict(test[FEATS]))

# Ensemble (simple average)
ens_pred = (xgb_pred + lgb_pred) / 2

def mae(y, p): return mean_absolute_error(y, p)

print('=' * 45)
print(f'{"Model":<12} {"MAE (MW)":>10} {"Train Time":>12}')
print('-' * 45)
print(f'{"XGBoost":<12} {mae(test["solar_mw"], xgb_pred):>10.3f} {xgb_time:>10.1f}s')
print(f'{"LightGBM":<12} {mae(test["solar_mw"], lgb_pred):>10.3f} {lgb_time:>10.1f}s')
print(f'{"Ensemble":<12} {mae(test["solar_mw"], ens_pred):>10.3f} {"—":>11}')
print('=' * 45)
print('\nEnsemble มักดีกว่าตัวใดตัวหนึ่ง เพราะ error หักล้างกัน')

---
## Technique 6: Optuna — Auto Hyperparameter Tuning

**Grid Search** → ลองทุก combination → O(n^k) ช้ามาก  
**Optuna** → Bayesian optimization → เรียนจาก trial ที่แล้ว → หา good params ใน 50-100 trials

```
Trial 1: max_depth=4 → MAE=0.45
Trial 2: max_depth=8 → MAE=0.52  (แย่ลง → Optuna หลีกเลี่ยงโซนนี้)
Trial 3: max_depth=5 → MAE=0.41  (ดีขึ้น → explore รอบๆ)
...
Trial 50: max_depth=6, lr=0.03 → MAE=0.38  ← best
```

In [ ]:
def objective(trial):
    """Optuna objective — minimize CV MAE"""
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 500),
        'max_depth':        trial.suggest_int('max_depth', 3, 10),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'random_state': 42, 'n_jobs': -1, 'verbosity': 0
    }

    tscv = TimeSeriesSplit(n_splits=3)
    scores = []
    for train_idx, val_idx in tscv.split(X):
        m = xgb.XGBRegressor(**params)
        m.fit(X.iloc[train_idx], y.iloc[train_idx])
        pred = np.maximum(0, m.predict(X.iloc[val_idx]))
        scores.append(mean_absolute_error(y.iloc[val_idx], pred))
    return np.mean(scores)


study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=30, show_progress_bar=True)
# เพิ่มเป็น n_trials=100 สำหรับผลที่ดีกว่า

print('\nBest params:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')
print(f'Best CV MAE: {study.best_value:.3f} MW')

In [ ]:
# Plot optimization history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

# Trial MAE over time
trials_df = study.trials_dataframe()
ax1.plot(trials_df['number'], trials_df['value'], 'o-', alpha=0.5, ms=4)
ax1.axhline(study.best_value, color='red', ls='--', label=f'Best={study.best_value:.3f}')
ax1.set_xlabel('Trial'); ax1.set_ylabel('MAE (MW)')
ax1.set_title('Optuna: MAE per trial')
ax1.legend(); ax1.grid(alpha=0.3)

# Parameter importance (Optuna built-in)
importance = optuna.importance.get_param_importances(study)
params_names = list(importance.keys())
params_vals  = list(importance.values())
ax2.barh(params_names[::-1], params_vals[::-1], color='steelblue')
ax2.set_title('Optuna: Parameter Importance')
ax2.set_xlabel('Importance')

plt.tight_layout(); plt.show()

In [ ]:
# Train final model ด้วย best params
best_model = xgb.XGBRegressor(**study.best_params, random_state=42, n_jobs=-1, verbosity=0)
best_model.fit(train[FEATS], train['solar_mw'])
best_pred = np.maximum(0, best_model.predict(test[FEATS]))

# เปรียบเทียบ default vs tuned
default_mae = mae(test['solar_mw'], xgb_pred)
tuned_mae   = mae(test['solar_mw'], best_pred)
improve     = (default_mae - tuned_mae) / default_mae * 100

print(f'Default XGBoost MAE: {default_mae:.3f} MW')
print(f'Tuned   XGBoost MAE: {tuned_mae:.3f} MW')
print(f'Improvement:         {improve:.1f}%')

---
## สรุป: เลือกเทคนิคอะไรก่อน?

```
1. ใช้ Fourier features เพิ่มทันที       → ง่าย ได้ผลทุกครั้ง
2. ใช้ TimeSeriesSplit CV                → ห้ามข้ามเด็ดขาด
3. Ensemble XGBoost + LightGBM           → +1-3% accuracy ฟรี
4. Optuna tune                           → ดีที่สุดถ้ามีเวลา
5. SHAP → debug + pitching               → อธิบาย model ให้กรรมการ
6. Permutation importance → feature cut  → ลด noise
```

**สำหรับ competition:**
- เตรียมก่อนแข่ง: Fourier + Optuna + Ensemble (train ล่วงหน้า)
- วันแข่ง: แค่รัน inference + submission function
- Pitching: SHAP plot อธิบาย model ได้น่าประทับใจ